https://knaaptime.com/urban_analysis/02_interpolation/dasymetric.html

#### interpolation of resident numbers from 2010 to 2021

Using Target Density Weighting (TDW) method found in the paper `Data-enriched Interpolation for Temporally Consistent Population Compositions`

1. Calculate subzone densities:
$Density = \frac{residents\ in\ subzone}{land area}$

2. Establish weight ratio:

Assume the subzone population is proportional to the density of the year's total population. Calculate a `density ratio` for each subzone

$Ratio = \frac{Subzone\ Density}{National\ Average\ Density}$

3. Temporal Interpolation of Weights:

As we only have population data for 2010, 2015 and 2020, need to estimate the density weights for the missing years.

Target Density Weighting method assumes that the subzone's density ratio moves linearly over time.

4. Interpolate:

Multiply the annual national counts for each age group by the calculated weights. 

#### Interpolate resident numbers by age group
Census data for 2020 follows 2019 masterplan for geospatial data, 2015 follows 2014 masterplan, 2010 follows 2008 masterplan

In [1]:
import pandas as pd
import numpy as np
import geopandas as gpd
from pathlib import Path
import os
from shapely.wkt import loads

In [2]:
CURRENT_DIRECTORY =  os.getcwd()
SRC_DIRECTORY = Path(CURRENT_DIRECTORY).parents[1]
print(SRC_DIRECTORY)

BASE_DATASET_PATH = Path(SRC_DIRECTORY).parents[0]
BASE_DATASET_PATH = BASE_DATASET_PATH / "datasets"
print(BASE_DATASET_PATH)

/Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/src
/Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/datasets


In [3]:
# get subzone data with their corresponding areas in km^2
def get_subzone_data(year: int):
    """
    Args
    ------
    year: int
        year of the subzones you want (2019, 2014, 2008)

    Returns
    ------
    Subzone dataframe containing the subzone names, planning area names and areas of points of interest.
    """

    year = str(year)

    subzones_filepath = Path(BASE_DATASET_PATH / f"singapore_data/data_gov/masterplan_{year}/subzone_boundary_{year}.gpkg")
    subzones_gpd = gpd.read_file(subzones_filepath)
    subzones_gpd = subzones_gpd.map(lambda s: s.lower() if isinstance(s, str) else s)
    # lower() the column names of subzones_gpd
    subzones_gpd.columns = subzones_gpd.columns.str.lower()    

    # there is no landuse data found for 2008 masterplan, 
    # so not able to merge with the subzone_classification file obtained from extracting_landuse_type.ipynb
    if year == "2008":
        return subzones_gpd

    subzones_filepath = Path(BASE_DATASET_PATH / f"singapore_data/data_gov/masterplan_{year}/subzone_classifications_{year}.csv")
    subzones_csv = pd.read_csv(subzones_filepath)
    subzones_csv = subzones_csv.map(lambda s: s.lower() if isinstance(s, str) else s)

    subzones_with_geom = subzones_csv.merge(
        subzones_gpd[["subzone_n", "geometry"]],
        on = "subzone_n",
        how = "left"
    )

    missing_count = subzones_with_geom[subzones_with_geom.columns[-1]].isna().sum()
    if missing_count > 0:
        print(f"Warning: {missing_count} subzones did not find a match in the CSV.")

    return subzones_with_geom

In [ ]:
def get_demographic_data(year: int):

    year = str(year)

    demographic_filepath = Path(BASE_DATASET_PATH / f"singapore_data/cleaned_data/{year}_age_group_planning_area_subzone.xlsx")
    demographic_df = pd.read_excel(demographic_filepath, sheet_name = "subzone")

    demographic_df = demographic_df.rename(columns = {
        "subzone": "subzone_n",
        "planning_area": "pln_area_n"
    })

    # 2015 census data has different column names, standardise it
    if year == "2015":
        old_cols = [
            'total_0_4', 'total_5_9', 'total_10_14', 'total_15_19', 'total_20_24', 
            'total_25_29', 'total_30_34', 'total_35_39', 'total_40_44', 'total_45_49', 
            'total_50_54', 'total_55_59', 'total_60_64', 'total_65_69', 'total_70_74', 
            'total_75_79', 'total_80_84', 'total_85andover'
        ]

        mapping = {}
        for col in old_cols:
            # Remove 'total_' prefix and replace underscores with ' - '
            new_name = col.replace('total_', '').replace('_', ' - ')
            
            # Handle the special '85andover' case you mentioned
            if '85andover' in new_name:
                mapping[col] = '85 - 89' # Based on your specific request
            else:
                mapping[col] = new_name

        mapping["total_85andover"] = "85 & Over"

        demographic_df = demographic_df.rename(columns = mapping)

    demographic_df["pln_area_n"] = demographic_df["pln_area_n"].ffill()

    ## the subzones for punggol is named differently in the masterplan for 2010 and 2015, 
    # instead of eg: subzone 2, it is sz2, so renaming it
    if year == "2015" or year == "2010":
        demographic_df.loc[
            (demographic_df["subzone_n"] == "subzone 2") &
            (demographic_df["pln_area_n"] == "punggol"), 
            "subzone_n"
        ] = "sz2"
        demographic_df.loc[
            (demographic_df["subzone_n"] == "subzone 3") &
            (demographic_df["pln_area_n"] == "punggol"), 
            "subzone_n"
        ] = "sz3"
        demographic_df.loc[
            (demographic_df["subzone_n"] == "subzone 4") &
            (demographic_df["pln_area_n"] == "punggol"), 
            "subzone_n"
        ] = "sz4"
        demographic_df.loc[
            (demographic_df["subzone_n"] == "subzone 5") &
            (demographic_df["pln_area_n"] == "punggol"), 
            "subzone_n"
        ] = "sz5"

    return demographic_df

In [17]:
def convert_to_geodataframe(df):
    if not isinstance(df, gpd.GeoDataFrame):
        df = gpd.GeoDataFrame(df, geometry='geometry')

    # ensure the original data has the correct starting CRS (WGS84)
    # (Only needed if it isn't already set; usually .gpkg files have this metadata)
    if df.crs is None:
        df.set_crs(epsg=4326, inplace=True)

    # transform the entire GeoDataFrame to SVY21 (Meters)
    df_meters = df.to_crs(epsg=3414)

    return df_meters

# Step 1 : Calculate population density per subzone

In [33]:
# subzone_df = get_subzone_data(2008)
# subzone_df.head(3)

In [32]:
# # convert the Pandas DataFrame back to a GeoDataFrame
# # We explicitly pass the geometry column
# if not isinstance(subzone_df, gpd.GeoDataFrame):
#     subzone_df = gpd.GeoDataFrame(subzone_df, geometry='geometry')

# # ensure the original data has the correct starting CRS (WGS84)
# # (Only needed if it isn't already set; usually .gpkg files have this metadata)
# if subzone_df.crs is None:
#     subzone_df.set_crs(epsg=4326, inplace=True)

# # transform the entire GeoDataFrame to SVY21 (Meters)
# subzone_df_meters = subzone_df.to_crs(epsg=3414)

# # calculate area
# subzone_df['area_sqm'] = subzone_df_meters.geometry.area
# subzone_df['area_sqkm'] = subzone_df['area_sqm'] / 1_000_000

# subzone_areas = subzone_df[['subzone_n', 'pln_area_n', 'area_sqm', 'area_sqkm', "geometry"]].copy()

# # display(subzone_df[['subzone_n', 'area_sqkm']].head())
# display(subzone_areas)

In [31]:
# demographic_df = get_demographic_data(2010)
# demographic_df

#### calculate the density of each subzone
Density (resident per $km^2$) = $\frac{residents\ in\ subzone}{land\ area\ km^2}$

####  each set of census data has different recordings of age group numbers. I have chosen to preserve them instead of combining ages 65 and over. This is because granularity will help with the accuracy of the Target Density Weighting interpolation

In [8]:
def calculate_density(demographic_df, subzone_areas, year:int):
    
    year = str(year)

    if year == "2010":
        age_grp_columns = ['subzone_n', 'pln_area_n', 'total', '0 - 4', '5 - 9', '10 - 14',
       '15 - 19', '20 - 24', '25 - 29', '30 - 34', '35 - 39', '40 - 44',
       '45 - 49', '50 - 54', '55 - 59', '60 - 64', '65 & Over', 'total_above_60']
    
    if year == "2015":    
        age_grp_columns = ['subzone_n', 'pln_area_n', 'total', '0 - 4', '5 - 9', '10 - 14',
       '15 - 19', '20 - 24', '25 - 29', '30 - 34', '35 - 39', '40 - 44',
       '45 - 49', '50 - 54', '55 - 59', '60 - 64', '65 - 69', '70 - 74',
       '75 - 79', '80 - 84', '85 & Over', 'total_above_60']
        
    if year == "2020":
        age_grp_columns = ['subzone_n', 'pln_area_n', 'total', '0 - 4', '5 - 9', '10 - 14',
       '15 - 19', '20 - 24', '25 - 29', '30 - 34', '35 - 39', '40 - 44',
       '45 - 49', '50 - 54', '55 - 59', '60 - 64', '65 - 69', '70 - 74',
       '75 - 79', '80 - 84', '85 - 89', '90 & over', 'total_above_60']
        
    demographic_df = demographic_df[age_grp_columns].copy()

    # standardize casing and remove hidden whitespace
    subzone_areas['subzone_n'] = subzone_areas['subzone_n'].astype(str).str.strip().str.lower()
    demographic_df['subzone_n'] = demographic_df['subzone_n'].astype(str).str.strip().str.lower()

    density_df = subzone_areas.merge(
        demographic_df,
        on = ["subzone_n", "pln_area_n"],
        how = "left"
    )
    
    for age_group in age_grp_columns[2:]:
        new_column_name = "density_" + age_group

        density_df[f"density_{age_group}"] =  density_df[age_group] / density_df["area_sqkm"]

    return density_df
    

In [ ]:
# density_df = calculate_density(demographic_df, subzone_areas, 2010)
# # fill NaN results with 0 (0 population data for that subzone)
# density_df = density_df.fillna(np.nan)

# save_to_file = Path(BASE_DATASET_PATH / f"singapore_data/data_gov/masterplan_2008/pop_density_2010.csv")
# density_df.to_csv(save_to_file)

In [22]:
# sanity check to see if merging worked
# checking if left join duplicated any subzone rows. 
# Note that some planning areas have the same subzone names. 
# And some subzone did not appear in the demographic dataset so their density columns will be 0 / NaN

def sanity_checks(density_df, subzone_areas, demographic_df):
    duplicate_mask = density_df.duplicated(subset=['subzone_n'], keep=False)

    # Get the unique names of those duplicates
    duplicate_names = density_df.loc[duplicate_mask, 'subzone_n'].unique()

    print("Planning areas with duplicates:")
    print(list(duplicate_names))

    ## checking if merging was successful. 
    # this checks which subzone and planning area pair from the demographic_df did not match with subzone_areas dataframe.
    subzone_areas['subzone_n'] = subzone_areas['subzone_n'].astype(str).str.strip().str.lower()
    # subzone_areas.to_csv("subzone_areas.csv")
    demographic_df['subzone_n'] = demographic_df['subzone_n'].astype(str).str.strip().str.lower()
    # demographic_df.to_csv("demographic_df.csv")

    check_df = subzone_areas[['subzone_n', 'pln_area_n']].merge(
        demographic_df[['subzone_n', 'pln_area_n']], 
        on=['subzone_n', 'pln_area_n'],
        how='outer', 
        indicator=True
    )

    failed_df = check_df[check_df['_merge'] == 'right_only'][['subzone_n', 'pln_area_n']]
    print(failed_df.to_string(index=False))


In [ ]:
# ## checking if merging was successful. 
# # this checks which subzone and planning area pair from the demographic_df did not match with subzone_areas dataframe.
# subzone_areas['subzone_n'] = subzone_areas['subzone_n'].astype(str).str.strip().str.lower()
# # subzone_areas.to_csv("subzone_areas.csv")
# demographic_df['subzone_n'] = demographic_df['subzone_n'].astype(str).str.strip().str.lower()
# # demographic_df.to_csv("demographic_df.csv")

# check_df = subzone_areas[['subzone_n', 'pln_area_n']].merge(
#     demographic_df[['subzone_n', 'pln_area_n']], 
#     on=['subzone_n', 'pln_area_n'],
#     how='outer', 
#     indicator=True
# )

# failed_df = check_df[check_df['_merge'] == 'right_only'][['subzone_n', 'pln_area_n']]
# print(failed_df.to_string(index=False))

# Step 2: Calculate Density Ratio per subzone

#### First we need to obtain the national average density for each age group

#### Calculating National Average Density
$\frac{Sum\ total\ resident}{Sum\ total\ area\ of\ Singapore}$

In [11]:
def calculate_national_average_density(density_df, year: int):
    """
    return the density of each age group across whole of Singapore
    """

    year = str(year)

    if year == "2010":
        age_cols = ['total', '0 - 4', '5 - 9', '10 - 14',
       '15 - 19', '20 - 24', '25 - 29', '30 - 34', '35 - 39', '40 - 44',
       '45 - 49', '50 - 54', '55 - 59', '60 - 64', '65 & Over', 'total_above_60']

    if year == "2015":    
        age_cols = ['total', '0 - 4', '5 - 9', '10 - 14',
       '15 - 19', '20 - 24', '25 - 29', '30 - 34', '35 - 39', '40 - 44',
       '45 - 49', '50 - 54', '55 - 59', '60 - 64', '65 - 69', '70 - 74',
       '75 - 79', '80 - 84', '85 & Over', 'total_above_60']

    if year == "2020":
        age_cols = ['total', '0 - 4', '5 - 9', '10 - 14',
       '15 - 19', '20 - 24', '25 - 29', '30 - 34', '35 - 39', '40 - 44',
       '45 - 49', '50 - 54', '55 - 59', '60 - 64', '65 - 69', '70 - 74',
       '75 - 79', '80 - 84', '85 - 89', '90 & over', 'total_above_60']

    # density_cols = [col for col in density_df.columns if "density" in col.lower()]

    total_area_all_subzones = density_df['area_sqkm'].sum()

    aggregated_results = []

    for col in age_cols:
        total_resident_count = density_df[col].sum()
        avg_density = total_resident_count / total_area_all_subzones
        
        aggregated_results.append({
            'age_group': col,
            'national_avg_density': avg_density
        })

    national_density_df = pd.DataFrame(aggregated_results)

    return national_density_df

In [30]:
# national_age_grp_density_df = calculate_national_average_density(density_df, 2010)
# national_age_grp_density_df

#### Calculate Density Ratio of each subzone
$Ratio = \frac{Subzone\ Density}{National\ Average\ Density}$

In [13]:
def calculate_density_ratio(density_df, national_age_grp_density_df):
    age_group_list = national_age_grp_density_df["age_group"].tolist()
    
    for age_group in age_group_list:

        national_average_density = national_age_grp_density_df.loc[
            national_age_grp_density_df["age_group"] == age_group,
            "national_avg_density"
        ].item()

        # if that age group is 0, set to np.nan to indicate that the national average was 0. 
        if national_average_density <= 0:
            density_df[f"density_ratio_{age_group}"] = np.nan

        else:
            density_df[f"density_ratio_{age_group}"] = density_df[f"density_{age_group}"] / national_average_density

    return density_df

In [ ]:
density_ratio_df = calculate_density_ratio(density_df, national_age_grp_density_df)

# export and check 
save_to_file = Path(BASE_DATASET_PATH / f"singapore_data/data_gov/masterplan_2008/pop_density_ratio_2010.csv")
density_ratio_df.to_csv(save_to_file)

# 3. Temporal Interpolation of Weights

In [34]:
read_from_file_2010 = Path(BASE_DATASET_PATH / f"singapore_data/cleaned_data/subzone_density_ratios/pop_density_ratio_2010.csv")
read_from_file_2015 = Path(BASE_DATASET_PATH / f"singapore_data/cleaned_data/subzone_density_ratios/pop_density_ratio_2015.csv")
read_from_file_2020 = Path(BASE_DATASET_PATH / f"singapore_data/cleaned_data/subzone_density_ratios/pop_density_ratio_2020.csv")

density_ratio_df_2010 = pd.read_csv(read_from_file_2010)
density_ratio_df_2015 = pd.read_csv(read_from_file_2015)
density_ratio_df_2020 = pd.read_csv(read_from_file_2020)


In [ ]:
# plot the density ratio changes for each planning area to check if there is any sudden spike or decrease in density ratio
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# 1. Prepare the temporal data (Long Format)
# We assume the columns are named 'subzone_n', 'pln_area_n', and 'density_ratio_total'
d10 = density_ratio_df_2010[['subzone_n', 'pln_area_n', 'density_ratio_total']].rename(columns={'density_ratio_total': '2010'})
d15 = density_ratio_df_2015[['subzone_n', 'pln_area_n', 'density_ratio_total']].rename(columns={'density_ratio_total': '2015'})
d20 = density_ratio_df_2020[['subzone_n', 'pln_area_n', 'density_ratio_total']].rename(columns={'density_ratio_total': '2020'})

# Merge and Melt
merged = d10.merge(d15, on=['subzone_n', 'pln_area_n'], how='outer')
merged = merged.merge(d20, on=['subzone_n', 'pln_area_n'], how='outer')
df_long = merged.melt(id_vars=['subzone_n', 'pln_area_n'], var_name='Year', value_name='Ratio')
df_long['Year'] = df_long['Year'].astype(int)
df_long = df_long.fillna(0)

# Create a directory to store the images if it doesn't exist
output_dir = "planning_area_plots"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# 2. Loop through unique Planning Areas
unique_areas = df_long['pln_area_n'].unique()

for area in unique_areas:
    # Filter data for this specific area
    subset = df_long[df_long['pln_area_n'] == area]
    
    # Skip if area is NaN or empty
    if pd.isna(area) or subset.empty:
        continue

    # Create the plot
    plt.figure(figsize=(12, 7))
    sns.lineplot(data=subset, x='Year', y='Ratio', hue='subzone_n', marker='o', linewidth=2)
    
    # Formatting
    plt.title(f"Temporal Density Ratio: {area.upper()}", fontsize=15)
    plt.ylabel("Density Ratio (Subzone / National)")
    plt.xlabel("Census Year")
    plt.xticks([2010, 2015, 2020])
    plt.ylim(0, subset['Ratio'].max() * 1.2) # Adjust height based on local max
    
    # Legend handling (move outside if many subzones)
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title="Subzones")
    plt.grid(True, linestyle=':', alpha=0.6)
    
    # Save the file
    # Replace spaces with underscores for cleaner filenames
    filename = f"{area.replace(' ', '_').lower()}_trend.png"
    plt.savefig(os.path.join(output_dir, filename), bbox_inches='tight', dpi=150)
    
    # Close the plot to free up memory during the loop
    plt.close()

print(f"Successfully saved {len(unique_areas)} plots to the '{output_dir}' folder.")

/var/folders/b8/1t_kfq0s27zbtp79vq8klgfm0000gn/T/ipykernel_30099/1059443842.py:44: UserWarning: Attempting to set identical low and high ylims makes transformation singular; automatically expanding.
  plt.ylim(0, subset['Ratio'].max() * 1.2) # Adjust height based on local max
/var/folders/b8/1t_kfq0s27zbtp79vq8klgfm0000gn/T/ipykernel_30099/1059443842.py:44: UserWarning: Attempting to set identical low and high ylims makes transformation singular; automatically expanding.
  plt.ylim(0, subset['Ratio'].max() * 1.2) # Adjust height based on local max
/var/folders/b8/1t_kfq0s27zbtp79vq8klgfm0000gn/T/ipykernel_30099/1059443842.py:44: UserWarning: Attempting to set identical low and high ylims makes transformation singular; automatically expanding.
  plt.ylim(0, subset['Ratio'].max() * 1.2) # Adjust height based on local max
/var/folders/b8/1t_kfq0s27zbtp79vq8klgfm0000gn/T/ipykernel_30099/1059443842.py:44: UserWarning: Attempting to set identical low and high ylims makes transformation singu

Successfully saved 56 plots to the 'planning_area_plots' folder.


/var/folders/b8/1t_kfq0s27zbtp79vq8klgfm0000gn/T/ipykernel_30099/1059443842.py:44: UserWarning: Attempting to set identical low and high ylims makes transformation singular; automatically expanding.
  plt.ylim(0, subset['Ratio'].max() * 1.2) # Adjust height based on local max


As we only have population data for 2010, 2015 and 2020, need to estimate the density weights for the missing years.

Target Density Weighting method assumes that the subzone's density ratio moves linearly over time. So to interpolate for the density ratio changes from 2011 to 2020, we draw a stright line between the 2010 - 2015 density ratio and 2015 - 2020 density ratio. 

In [ ]:
def interpolate_weights_and_estimate(anchor_ratio_df, national_annual_df, subzone_areas):
    # 1. Create a backbone for all years 2010-2021
    years = np.arange(2010, 2022)
    subzones = anchor_ratio_df['subzone_n'].unique()
    age_groups = anchor_ratio_df['AgeGroup'].unique()
    
    full_idx = pd.MultiIndex.from_product([subzones, age_groups, years], 
                                          names=['subzone_n', 'AgeGroup', 'Year'])
    df_timeline = pd.DataFrame(index=full_idx).reset_index()
    
    # 2. Merge anchor ratios (2010, 2015, 2020) and interpolate
    df_timeline = pd.merge(df_timeline, anchor_ratio_df, on=['subzone_n', 'AgeGroup', 'Year'], how='left')
    
    # Sort and interpolate ratios per subzone/age group
    df_timeline['Interp_Ratio'] = df_timeline.groupby(['subzone_n', 'AgeGroup'])['WeightingRatio'].transform(
        lambda x: x.interpolate(method='linear', limit_direction='both')
    )
    
    # 3. Merge with subzone areas and national annual data
    df_timeline = pd.merge(df_timeline, subzone_areas, on='subzone_n')
    final_df = pd.merge(df_timeline, national_annual_df, on=['Year', 'AgeGroup'])
    
    # 4. Final TDW Calculation 
    # National Avg Density = NationalCount / Total_Singapore_Area
    final_df['Nat_Avg_Density'] = final_df['NationalCount'] / final_df['Total_SG_Area']
    
    # Estimated Count = (Ratio * Area) * National Density
    final_df['Estimated_Residents'] = (final_df['Interp_Ratio'] * final_df['area_sqkm']) * final_df['Nat_Avg_Density']
    
    return final_df

# Example usage:
# result_df = interpolate_weights_and_estimate(anchor_ratios, national_data, subzone_areas)

In [ ]:
def perform_step_1_and_2():
    
    # dictionary of the available demographic data years and their corresponding subzone data years
    anchor_years = {2010: 2008,
                    2015: 2014,
                    2020: 2019}

    for demographic_year, subzone_year in anchor_years.items():
        subzone_df = get_subzone_data(subzone_year)
        subzone_df_meters = convert_to_geodataframe(subzone_df)

        # calculate area
        subzone_df['area_sqm'] = subzone_df_meters.geometry.area
        subzone_df['area_sqkm'] = subzone_df['area_sqm'] / 1_000_000
        subzone_areas = subzone_df[['subzone_n', 'pln_area_n', 'area_sqm', 'area_sqkm', "geometry"]].copy()

        demographic_df = get_demographic_data(demographic_year)
        density_df = calculate_density(demographic_df, subzone_areas, demographic_year)
        # fill NaN results with 0 (0 population data for that subzone)
        density_df = density_df.fillna(np.nan)

        print(f"\nPerforming sanity check for {demographic_year}")
        sanity_checks(density_df, subzone_areas, demographic_df)

        national_age_grp_density_df = calculate_national_average_density(density_df, demographic_year)
        density_ratio_df = calculate_density_ratio(density_df, national_age_grp_density_df)
        
        save_to_file = Path(BASE_DATASET_PATH / f"singapore_data/cleaned_data/subzone_density_ratios/pop_density_ratio_{demographic_year}.csv")
        density_ratio_df.to_csv(save_to_file)

In [29]:
perform_step_1_and_2()


Performing sanity check for 2010
Planning areas with duplicates:
['seletar', 'boon lay', 'hong kah', 'central', 'pang sua', 'trafalgar', 'marymount', 'tanjong pagar', 'marina east']
subzone_n      pln_area_n
    total      ang mo kio
    total           bedok
    total          bishan
    total     bukit batok
    total     bukit merah
    total   bukit panjang
    total     bukit timah
    total          changi
    total   choa chu kang
    total        clementi
    total   downtown core
    total         geylang
    total         hougang
    total     jurong east
    total     jurong west
    total         kallang
    total          mandai
    total   marine parade
    total          newton
    total          novena
    total          others
    total          outram
    total       pasir ris
    total         punggol
    total      queenstown
    total    river valley
    total          rochor
    total       sembawang
    total        sengkang
    total       serangoon
    total s